# AML Detection -- ExSTraQt-style pipeline

```
CSV
  |
  v
data_processing.py        (unchanged -- clean, encode, sort chronologically)
  |
  v
graph_construction.py     (transaction graph AND account graph)
  |
  v
feature_engineering.py    (existing behavioral/temporal/pair-history features)
community_detection.py    (Leiden + random-walk communities, on the account graph)
flow_features.py          (dispense/sink/passthrough + temporal flow)
  |
  v
feature_merge.py           (behavior + flow + community -> one dataframe)
  |
  v
model.py / train.py        (one XGBoost)
  |
  v
explain.py                 (SHAP)  +  aggregation.py (dashboard exports)
```
See `../README.md` for the full design writeup, including the `payment_format_idx` finding.

In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
import plotly.express as px

import config
from data_processing import load_and_clean
from graph_construction import build_transaction_graph
from feature_engineering import engineer_all_features
from base_feature_columns import NUMERIC_FEATURE_COLUMNS, CATEGORICAL_FEATURE_COLUMNS
import train as T
from aggregation import build_transaction_view, aggregate_accounts

COMBINE_WITH_BASE_FEATURES = True
RESTRICT_TO_ACCOUNTS = config.RESTRICT_TO_ACCOUNTS  # see config.py -- None = every account

In [ ]:
from pathlib import Path

# ==========================
# DATA PATHS
# ==========================

TRAIN_CSV = Path("/kaggle/input/datasets/aadityapandey1/aml-xgb/HI-Small_Trans.csv")
TEST_CSV = Path("/kaggle/input/datasets/aadityapandey1/aml-xgb/LI-Small_Trans.csv")

# ==========================
# OUTPUT ROOT
# ==========================

OUTPUT_ROOT = Path("/kaggle/working")

CACHE_DIR = OUTPUT_ROOT / "cache"

GRAPH_CACHE_DIR = CACHE_DIR / "graphs"
COMMUNITY_CACHE_DIR = CACHE_DIR / "communities"
FEATURE_CACHE_DIR = CACHE_DIR / "features"
MODEL_CACHE_DIR = CACHE_DIR / "models"
EXPLAIN_CACHE_DIR = CACHE_DIR / "explainability"

EXPORT_DIR = OUTPUT_ROOT / "exports"

for p in [
    GRAPH_CACHE_DIR,
    COMMUNITY_CACHE_DIR,
    FEATURE_CACHE_DIR,
    MODEL_CACHE_DIR,
    EXPLAIN_CACHE_DIR,
    EXPORT_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

In [ ]:
df_train, vocabs = load_and_clean(TRAIN_CSV))
print(f"Train: {len(df_train):,} transactions, laundering rate {df_train['label'].mean():.3%}")

if COMBINE_WITH_BASE_FEATURES:
    _, src, dst, preds, succs = build_transaction_graph(df_train)
    df_train = engineer_all_features(df_train, preds, succs, src, dst)
    base_numeric, base_categorical = NUMERIC_FEATURE_COLUMNS, CATEGORICAL_FEATURE_COLUMNS
else:
    base_numeric, base_categorical = [], []

## Train
Community/flow/anomaly features are cached under `../cache/` at every stage (see `utils.py`) -- reruns after this point are fast unless you change the underlying transactions or `RESTRICT_TO_ACCOUNTS`.

In [ ]:
result = T.train(
    GRAPH_CACHE_DIR, COMMUNITY_CACHE_DIR, FEATURE_CACHE_DIR,
    df_train, val_frac=0.15,
    restrict_to_accounts=RESTRICT_TO_ACCOUNTS,
    base_numeric_columns=base_numeric, base_categorical_columns=base_categorical,
)
model, threshold = result["model"], result["threshold"]
print(result["metrics"])

In [ ]:
importance = model.feature_importance(top_n=25)
importance.to_csv(EXPORT_DIR / "feature_importance.csv", index=False)
px.bar(importance.sort_values("importance"), x="importance", y="feature", orientation="h",
       title="Feature importance (single ExSTraQt-style XGBoost)")

## SHAP: why was any ONE transaction flagged?
See `explain.py`. `model.feature_importance()` above is global; this is per-transaction.

In [ ]:
import explain
import feature_merge as fm

X_val = fm.prepare_feature_frame(result["df_val_joined"], result["feature_columns"]["numeric"], result["feature_columns"]["categorical"])
explainer = explain.build_explainer(model.booster_)
display(explain.summary(explainer, X_val, max_display=15))

flagged_idx = result["df_val_joined"].index[result["val_probs"] >= threshold]
if len(flagged_idx):
    display(explain.explain_transaction(explainer, X_val, flagged_idx[0]))

## Held-out file + dashboard export

In [ ]:
if TEST_CSV.exists():
    df_test, _ = load_and_clean(str(TEST_CSV), vocabs=vocabs)
    if COMBINE_WITH_BASE_FEATURES:
        _, src_t, dst_t, preds_t, succs_t = build_transaction_graph(df_test)
        df_test = engineer_all_features(df_test, preds_t, succs_t, src_t, dst_t)

    holdout = T.score_holdout(GRAPH_CACHE_DIR, COMMUNITY_CACHE_DIR,FEATURE_CACHE_DIR, 
                              model, df_train, df_test, result["feature_columns"], threshold,
                               restrict_to_accounts=RESTRICT_TO_ACCOUNTS)
    print(holdout["metrics"])

    transaction_view = build_transaction_view(holdout["df_holdout_joined"], holdout["probs"], threshold=threshold)
    account_view = aggregate_accounts(holdout["df_holdout_joined"], holdout["probs"], threshold=threshold)
    transaction_view.to_csv(EXPORT_DIR / "transaction_view.csv", index=False)
    account_view.to_csv(EXPORT_DIR / "account_view.csv", index=False)
    print("Wrote transaction_view.csv / account_view.csv / feature_importance.csv to data/exports/")
else:
    print(f"No held-out file at {TEST_CSV} -- skipping.")

In [ ]:
T.save_model(model, str(MODEL_CACHE_DIR / "exstraqt_model.json"))
print("Saved model to", MODEL_CACHE_DIR / "exstraqt_model.json")